# 🌟 3D Gaussian Splatting (3DGS) — Colab 자동 실험 노트북

[![GitHub](https://img.shields.io/badge/GitHub-jhm6944%2FGenTest-black?logo=github)](https://github.com/jhm6944/GenTest)
[![Paper](https://img.shields.io/badge/Paper-ACM%20TOG%202023-blue)](https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/3d_gaussian_splatting_high.pdf)

이 노트북은 **[jhm6944/GenTest](https://github.com/jhm6944/GenTest)** 레포지토리를 Colab에서 자동으로 클론하고,  
3DGS 환경 설치 → 데이터 준비 → 학습 → 평가까지 **원스톱**으로 실행합니다.

---
### 📋 실행 순서
1. GPU 확인
2. 레포지토리 클론 (GitHub)
3. 의존성 설치
4. CUDA 서브모듈 빌드
5. 데이터셋 다운로드
6. 학습 실행
7. 렌더링 & 평가
8. 결과 시각화

> ⚠️ **런타임 → 런타임 유형 변경 → T4 GPU** 로 설정 후 실행하세요!

---
## ⚙️ STEP 0: 설정값 (여기만 수정하세요)

In [ ]:
# ============================================================
# 🔧 실험 설정 — 필요에 따라 수정하세요
# ============================================================

# GitHub 레포지토리 (공개 레포는 토큰 불필요)
GITHUB_REPO = "https://github.com/jhm6944/GenTest.git"
REPO_BRANCH = "main"

# 데이터셋 선택
# 'tandt'  : Tanks & Temples (truck, train 씬)
# 'db'     : Deep Blending (drjohnson, playroom 씬)
# 'custom' : 직접 업로드한 데이터
DATASET_TYPE = "tandt"
SCENE_NAME   = "truck"       # tandt: truck / train, db: drjohnson / playroom

# 학습 파라미터
ITERATIONS      = 7000       # 빠른 테스트: 7000 / 풀 학습: 30000
SH_DEGREE       = 3          # 구면 조화 차수 (0~3)
EVAL            = True       # True = train/test split 평가
OPTIMIZER_TYPE  = "default"  # 'default' or 'sparse_adam' (가속화)
WHITE_BG        = False      # NeRF Synthetic 데이터는 True
RESOLUTION      = 2          # 1=원본, 2=1/2, 4=1/4 (VRAM 부족 시 높이기)

# 출력 경로
OUTPUT_DIR = "/content/output"

print("✅ 설정 완료!")
print(f"   레포: {GITHUB_REPO}")
print(f"   씬:   {SCENE_NAME} ({DATASET_TYPE})")
print(f"   반복: {ITERATIONS} iters, SH degree: {SH_DEGREE}")

---
## 🖥️ STEP 1: GPU 확인

In [ ]:
import subprocess, sys

def run(cmd, **kwargs):
    """명령어 실행 + 출력 표시"""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kwargs)
    if result.stdout: print(result.stdout)
    if result.returncode != 0 and result.stderr:
        print("[STDERR]", result.stderr[:500])
    return result

print("="*50)
print("GPU 정보")
print("="*50)
run("nvidia-smi")

import torch
print(f"\nPyTorch 버전: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU 이름: {torch.cuda.get_device_name(0)}")
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM: {vram:.1f} GB")
    if vram < 8:
        print("⚠️  VRAM 부족! RESOLUTION=4 또는 ITERATIONS를 줄이는 것을 권장")
    elif vram < 16:
        print("⚠️  VRAM 여유 없음. RESOLUTION=2 권장")
    else:
        print("✅ VRAM 충분!")
else:
    print("❌ GPU 없음! 런타임 → 런타임 유형 변경 → GPU 선택 후 재시작하세요")
    sys.exit()

---
## 📦 STEP 2: 레포지토리 클론 (GitHub → Colab)

In [ ]:
import os

REPO_DIR = "/content/GenTest"

if os.path.exists(REPO_DIR):
    print("📁 레포 이미 존재 → 최신 코드로 업데이트 중...")
    run(f"cd {REPO_DIR} && git pull origin {REPO_BRANCH}")
else:
    print(f"📥 GitHub에서 클론 중: {GITHUB_REPO}")
    run(f"git clone -b {REPO_BRANCH} --depth=1 {GITHUB_REPO} {REPO_DIR}")

# submodules 초기화
print("\n🔗 서브모듈 초기화...")
run(f"cd {REPO_DIR} && git submodule update --init --recursive")

# 작업 디렉토리 이동
os.chdir(REPO_DIR)
print(f"\n✅ 현재 디렉토리: {os.getcwd()}")
run("ls -la")

---
## 🔧 STEP 3: 의존성 설치

In [ ]:
print("📦 Python 패키지 설치 중...")

packages = [
    "plyfile",
    "tqdm",
    "lpips",
    "scipy",
    "imageio",
    "opencv-python-headless",
]

for pkg in packages:
    print(f"  → {pkg} 설치 중...")
    run(f"pip install -q {pkg}")

# torchvision 버전 확인 (Colab에 이미 설치됨)
import torchvision
print(f"\n✅ torchvision: {torchvision.__version__}")

print("\n✅ 패키지 설치 완료!")

---
## ⚡ STEP 4: CUDA 서브모듈 빌드
> 처음 실행 시 5~10분 소요됩니다

In [ ]:
import os
os.chdir(REPO_DIR)

print("🔨 diff-gaussian-rasterization 빌드 중... (5~10분 소요)")
result = run("pip install submodules/diff-gaussian-rasterization")
if result.returncode == 0:
    print("✅ diff-gaussian-rasterization 빌드 완료!")
else:
    print("❌ 빌드 실패! 로그를 확인하세요")

print("\n🔨 simple-knn 빌드 중...")
result = run("pip install submodules/simple-knn")
if result.returncode == 0:
    print("✅ simple-knn 빌드 완료!")
else:
    print("❌ 빌드 실패!")

print("\n🔨 fused-ssim 빌드 중... (선택사항, 학습 가속화용)")
result = run("pip install submodules/fused-ssim")
if result.returncode == 0:
    print("✅ fused-ssim 빌드 완료!")
else:
    print("⚠️  fused-ssim 빌드 실패 (학습 가속화 비활성화됨, 계속 진행)")

# 빌드 검증
print("\n🔍 빌드 검증...")
try:
    from diff_gaussian_rasterization import GaussianRasterizationSettings
    print("✅ diff_gaussian_rasterization import 성공!")
except ImportError as e:
    print(f"❌ import 실패: {e}")

try:
    from simple_knn._C import distCUDA2
    print("✅ simple_knn import 성공!")
except ImportError as e:
    print(f"❌ import 실패: {e}")

---
## 📁 STEP 5: 데이터셋 준비

In [ ]:
import os

DATA_DIR = "/content/data"
os.makedirs(DATA_DIR, exist_ok=True)

if DATASET_TYPE == "tandt":
    # Tanks & Temples + Deep Blending 데이터셋
    DATASET_URL = "https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/datasets/input/tandt_db.zip"
    ZIP_PATH = "/content/tandt_db.zip"

    if not os.path.exists(f"{DATA_DIR}/tandt"):
        print(f"📥 Tanks & Temples 데이터셋 다운로드 중... (약 650MB, 수 분 소요)")
        run(f"wget -q --show-progress -O {ZIP_PATH} {DATASET_URL}")
        print("📦 압축 해제 중...")
        run(f"unzip -q {ZIP_PATH} -d {DATA_DIR}")
        run(f"rm {ZIP_PATH}")
        print("✅ 데이터셋 준비 완료!")
    else:
        print("✅ 데이터셋 이미 존재!")

    SCENE_PATH = f"{DATA_DIR}/tandt/{SCENE_NAME}"

elif DATASET_TYPE == "db":
    DATASET_URL = "https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/datasets/input/tandt_db.zip"
    ZIP_PATH = "/content/tandt_db.zip"

    if not os.path.exists(f"{DATA_DIR}/db"):
        print("📥 Deep Blending 데이터셋 다운로드 중...")
        run(f"wget -q --show-progress -O {ZIP_PATH} {DATASET_URL}")
        run(f"unzip -q {ZIP_PATH} -d {DATA_DIR}")
        run(f"rm {ZIP_PATH}")
    SCENE_PATH = f"{DATA_DIR}/db/{SCENE_NAME}"

elif DATASET_TYPE == "custom":
    # Google Drive에서 업로드한 커스텀 데이터
    from google.colab import drive
    drive.mount('/content/drive')
    SCENE_PATH = "/content/drive/MyDrive/my_3dgs_data"  # 경로 수정 필요
    print(f"📁 커스텀 데이터 경로: {SCENE_PATH}")

# 데이터셋 구조 확인
print(f"\n📁 씬 경로: {SCENE_PATH}")
if os.path.exists(SCENE_PATH):
    run(f"ls -la {SCENE_PATH}/")
    run(f"ls -la {SCENE_PATH}/sparse/0/ 2>/dev/null || echo '(sparse 폴더 없음)'")
else:
    print(f"❌ 씬 경로가 존재하지 않습니다: {SCENE_PATH}")

---
## 🚀 STEP 6: 3DGS 학습 실행

In [ ]:
import os, subprocess, time
os.chdir(REPO_DIR)

# 출력 디렉토리
MODEL_OUTPUT = f"{OUTPUT_DIR}/{SCENE_NAME}"
os.makedirs(MODEL_OUTPUT, exist_ok=True)

# 학습 명령어 구성
train_cmd = [
    f"python train.py",
    f"-s {SCENE_PATH}",
    f"-m {MODEL_OUTPUT}",
    f"--iterations {ITERATIONS}",
    f"--sh_degree {SH_DEGREE}",
    f"--save_iterations {ITERATIONS}",
    f"--test_iterations {ITERATIONS}",
    f"--resolution {RESOLUTION}",
    f"--optimizer_type {OPTIMIZER_TYPE}",
]

if EVAL:
    train_cmd.append("--eval")
if WHITE_BG:
    train_cmd.append("--white_background")

full_cmd = " ".join(train_cmd)

print("="*60)
print("🚀 3DGS 학습 시작!")
print("="*60)
print(f"명령어:\n  {full_cmd}")
print(f"\n예상 소요 시간:")
print(f"  7,000 iter  → 약 5~10분 (T4)")
print(f"  30,000 iter → 약 30~50분 (T4)")
print(f"  현재 설정:  {ITERATIONS} iter")
print("="*60)

start_time = time.time()

# 실시간 출력으로 학습 실행
process = subprocess.Popen(
    full_cmd, shell=True,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1
)

for line in process.stdout:
    print(line, end='', flush=True)

process.wait()
elapsed = time.time() - start_time

print("\n" + "="*60)
if process.returncode == 0:
    print(f"✅ 학습 완료! 소요 시간: {elapsed/60:.1f}분")
    print(f"📁 모델 저장 위치: {MODEL_OUTPUT}")
    run(f"ls -lh {MODEL_OUTPUT}/point_cloud/iteration_{ITERATIONS}/")
else:
    print(f"❌ 학습 실패! (종료코드: {process.returncode})")
print("="*60)

---
## 🎬 STEP 7: 렌더링 & 정량 평가

In [ ]:
import os
os.chdir(REPO_DIR)

print("🎬 렌더링 중...")
render_cmd = f"python render.py -m {MODEL_OUTPUT} --skip_train"
run(render_cmd)

if EVAL:
    print("\n📊 정량 평가 중 (PSNR / SSIM / LPIPS)...")
    metrics_cmd = f"python metrics.py -m {MODEL_OUTPUT}"
    run(metrics_cmd)

    # 결과 파일 읽기
    import json, glob
    result_files = glob.glob(f"{MODEL_OUTPUT}/results.json")
    if result_files:
        with open(result_files[0]) as f:
            results = json.load(f)
        print("\n" + "="*50)
        print("📊 평가 결과")
        print("="*50)
        for iter_key, metrics in results.items():
            print(f"\n  Iteration {iter_key}:")
            for k, v in metrics.items():
                if isinstance(v, float):
                    print(f"    {k:15s}: {v:.4f}")
        print("="*50)
else:
    print("ℹ️  EVAL=False 이므로 정량 평가를 건너뜁니다")

---
## 🖼️ STEP 8: 결과 시각화

In [ ]:
import glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from IPython.display import display

# 렌더링된 이미지 찾기
render_dir = f"{MODEL_OUTPUT}/train/ours_{ITERATIONS}/renders/"
if not os.path.exists(render_dir):
    render_dir = f"{MODEL_OUTPUT}/test/ours_{ITERATIONS}/renders/"

renders = sorted(glob.glob(f"{render_dir}/*.png"))[:6]
gt_dir = render_dir.replace("renders", "gt")
gts = sorted(glob.glob(f"{gt_dir}/*.png"))[:6]

if renders:
    n = min(len(renders), 3)
    fig, axes = plt.subplots(2, n, figsize=(6*n, 8))
    fig.suptitle(f"3DGS 학습 결과 — {SCENE_NAME} ({ITERATIONS} iters)", fontsize=16, fontweight='bold')

    for i in range(n):
        # 렌더링 이미지
        axes[0][i].imshow(mpimg.imread(renders[i]))
        axes[0][i].set_title(f"Rendered #{i+1}", fontsize=11)
        axes[0][i].axis('off')

        # Ground Truth
        if gts and i < len(gts):
            axes[1][i].imshow(mpimg.imread(gts[i]))
            axes[1][i].set_title(f"Ground Truth #{i+1}", fontsize=11)
        else:
            axes[1][i].text(0.5, 0.5, 'GT 없음', ha='center', va='center')
            axes[1][i].set_title(f"Ground Truth #{i+1}", fontsize=11)
        axes[1][i].axis('off')

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/result_visualization.png", dpi=150, bbox_inches='tight')
    plt.show()
    print(f"✅ 시각화 이미지 저장: {OUTPUT_DIR}/result_visualization.png")
else:
    print(f"⚠️ 렌더링 이미지를 찾을 수 없습니다: {render_dir}")

---
## 💾 STEP 9: 결과 저장 (Google Drive 선택사항)

In [ ]:
# Google Drive에 결과 저장 (선택사항)
SAVE_TO_DRIVE = False  # True로 변경하면 Drive에 저장

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_OUTPUT = f"/content/drive/MyDrive/3dgs_results/{SCENE_NAME}"
    os.makedirs(DRIVE_OUTPUT, exist_ok=True)

    print("📤 결과를 Google Drive에 저장 중...")
    # 모델 파일 (PLY) 저장
    ply_files = glob.glob(f"{MODEL_OUTPUT}/point_cloud/**/*.ply", recursive=True)
    for ply in ply_files:
        run(f"cp {ply} {DRIVE_OUTPUT}/")
        print(f"  → {ply} 저장 완료")

    # 시각화 저장
    run(f"cp {OUTPUT_DIR}/result_visualization.png {DRIVE_OUTPUT}/")

    print(f"\n✅ Drive 저장 완료: {DRIVE_OUTPUT}")
else:
    print("ℹ️  Drive 저장을 건너뜁니다 (SAVE_TO_DRIVE = False)")
    print(f"\n📁 현재 세션 내 결과 위치:")
    run(f"find {MODEL_OUTPUT} -name '*.ply' | head -5")
    print(f"\n⚠️ Colab 세션 종료 시 /content/ 데이터는 삭제됩니다")
    print(f"   Drive 저장을 원하면 위 SAVE_TO_DRIVE = True 로 변경하세요")

---
## 🔬 STEP 10: 하이퍼파라미터 실험 (선택사항)
> 여러 설정을 자동으로 비교 실험합니다

In [ ]:
# 자동 실험 비교 (실행하고 싶은 경우 RUN_SWEEP = True)
RUN_SWEEP = False

if RUN_SWEEP:
    import json, time
    os.chdir(REPO_DIR)

    # 비교할 실험 설정 목록
    experiments = [
        {"name": "baseline",       "iterations": 7000,  "sh_degree": 3, "optimizer": "default"},
        {"name": "sparse_adam",    "iterations": 7000,  "sh_degree": 3, "optimizer": "sparse_adam"},
        {"name": "low_sh",         "iterations": 7000,  "sh_degree": 1, "optimizer": "default"},
        {"name": "more_iters",     "iterations": 15000, "sh_degree": 3, "optimizer": "default"},
    ]

    sweep_results = []

    for exp in experiments:
        exp_output = f"{OUTPUT_DIR}/sweep_{exp['name']}"
        os.makedirs(exp_output, exist_ok=True)

        print(f"\n{'='*50}")
        print(f"🧪 실험: {exp['name']}")
        print(f"   iterations={exp['iterations']}, sh_degree={exp['sh_degree']}, optimizer={exp['optimizer']}")
        print(f"{'='*50}")

        cmd = (
            f"python train.py -s {SCENE_PATH} -m {exp_output} "
            f"--iterations {exp['iterations']} --sh_degree {exp['sh_degree']} "
            f"--optimizer_type {exp['optimizer']} "
            f"--save_iterations {exp['iterations']} --test_iterations {exp['iterations']} "
            f"--eval --resolution {RESOLUTION} --quiet"
        )

        start = time.time()
        result = run(cmd)
        elapsed = time.time() - start

        # 결과 수집
        run(f"python render.py -m {exp_output} --quiet")
        run(f"python metrics.py -m {exp_output}")

        results_file = f"{exp_output}/results.json"
        metrics_data = {}
        if os.path.exists(results_file):
            with open(results_file) as f:
                raw = json.load(f)
            for iter_key, m in raw.items():
                metrics_data = m

        sweep_results.append({
            "experiment": exp["name"],
            "time_min": elapsed / 60,
            **metrics_data
        })

    # 결과 비교 테이블 출력
    print("\n" + "="*70)
    print("📊 하이퍼파라미터 실험 비교 결과")
    print("="*70)
    print(f"{'실험':20s} {'PSNR':>8s} {'SSIM':>8s} {'LPIPS':>8s} {'시간(분)':>10s}")
    print("-"*70)
    for r in sweep_results:
        psnr  = r.get('PSNR',  0.0)
        ssim  = r.get('SSIM',  0.0)
        lpips = r.get('LPIPS', 0.0)
        t     = r.get('time_min', 0.0)
        print(f"{r['experiment']:20s} {psnr:8.3f} {ssim:8.4f} {lpips:8.4f} {t:10.1f}")
    print("="*70)

else:
    print("ℹ️  하이퍼파라미터 스윕을 건너뜁니다 (RUN_SWEEP = False)")
    print("   실험을 원하면 RUN_SWEEP = True 로 변경하세요")

---
## 📝 실험 완료!

### 결과 파일 위치
```
/content/output/<scene>/
├── point_cloud/
│   └── iteration_<N>/
│       └── point_cloud.ply   ← 학습된 3DGS 모델
├── train/  또는  test/
│   └── ours_<N>/
│       ├── renders/           ← 렌더링 이미지
│       └── gt/                ← Ground Truth
└── results.json               ← PSNR / SSIM / LPIPS 수치
```

### PLY 파일 다운로드
왼쪽 파일 탐색기 → `/content/output/` 폴더에서 `point_cloud.ply` 파일을 우클릭 → 다운로드

### PLY 파일 뷰어
- [SuperSplat](https://playcanvas.com/supersplat/editor) - 브라우저에서 바로 보기
- [Luma AI](https://lumalabs.ai/) - 고품질 웹 뷰어
- [SIBR Viewer](https://github.com/graphdeco-inria/gaussian-splatting) - 공식 데스크탑 뷰어

### 참고 링크
- 🔗 [jhm6944/GenTest GitHub](https://github.com/jhm6944/GenTest)
- 📄 [원본 논문](https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/3d_gaussian_splatting_high.pdf)
- 📖 [한국어 학습 가이드 (README_KR.md)](https://github.com/jhm6944/GenTest/blob/main/README_KR.md)